# 网站摘要器（幽默文案版）

## 练习目标（理念）

用 **OpenAI Chat Completions** 把一段「伪网页正文」压成短摘要：

- **输入**：粘贴的网站/个人主页文本（本例未做 HTTP 抓取，直接给 `website_text`）
- **输出**：2–3 句、轻度幽默、Markdown（不要代码围栏）
- **可控创意**：`temperature=0.5` 在稳定与俏皮之间折中

## 和本课 Day 1 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| `.env` + API Key | `load_dotenv` / `OPENAI_API_KEY` |
| `messages` | system 定文案风格，user 塞 CONTENT |
| `chat.completions.create` | `model="gpt-4o-mini"` |
| Notebook 展示 | `display(Markdown(...))` |

## 怎么跑

1. 配置 `.env` 中的 `OPENAI_API_KEY`
2. 从上到下运行；可改 `website_text` 或 system 语气后再跑生成格


In [18]:
# ========== 导入：后面摘要流程要用的库 ==========

# 标准库 os：读环境变量（Environment Variables）
import os
# load_dotenv：从 .env 载入密钥，避免把 OPENAI_API_KEY 写进笔记本
from dotenv import load_dotenv
# Markdown / display：在 Jupyter 里渲染模型返回的 Markdown 文本
from IPython.display import Markdown, display
# OpenAI：官方 SDK 客户端，用于 chat.completions
from openai import OpenAI


In [19]:
# ========== 环境 + 客户端 ==========

# 加载 .env；override=True 让文件值覆盖已有同名环境变量
load_dotenv(override=True)

# 读出 API Key（本格赋值后未再校验；密钥仍供 OpenAI() 默认读取）
api_key = os.getenv('OPENAI_API_KEY')

# 初始化客户端（默认使用环境变量里的 OPENAI_API_KEY）
openai = OpenAI()


In [20]:
# ========== Prompt + 待摘要正文 + messages ==========

# 系统提示：角色=文案（copywriter）；约束句数、语气、Markdown、禁止 code block（英文指令保留）
system_prompt = """
You are a copywriter.

Write a concise 2–3 sentence summary with a light, witty tone.
Use short, clear sentences.
Avoid fluff and repetition.

Respond in Markdown only.
Do not use code blocks.
"""

# 模拟「网站正文」：这里直接写死个人页文本，不发起 HTTP 请求
website_text = """
Hi, I'm Emmelie Johansson, a Junior Backend Developer specialising in Java and Agentic AI

About me
I’m a Junior Backend Developer at Exxeta in Leipzig, Germany, working primarily with Java and Agentic AI systems.

Before moving into tech, I worked for over a decade as a translator. In 2022, I transitioned into software engineering driven by a strong desire to grow and learn something new.

Since then, I’ve focused on building backend systems in Java and using AI to streamline software workflows.
"""

# 构造 Chat Completions 的 messages：system 定规矩，user 放任务 + CONTENT
messages = [
    {"role": "system", "content": system_prompt},
    {
        "role": "user",
        # f-string：把 website_text 嵌进英文 user 指令；忽略导航/重复项的要求写在 prompt 里
        "content": f"""
Summarize the following website content in 2–3 sentences.
Ignore repeated items, navigation text, and UI labels.

CONTENT:
{website_text}
"""
    }
]


In [21]:
# ========== 调用模型生成摘要 ==========

# chat.completions.create：一次非流式补全
response = openai.chat.completions.create(
    # 模型 id 保持 gpt-4o-mini（便宜、适合短摘要）
    model="gpt-4o-mini",
    # 上一格组好的 system + user
    messages=messages,
    # temperature：0.5 中等随机度，略活泼又不至于胡写
    temperature=0.5
)

# 从响应对象取出第一条候选的文本内容
summary = response.choices[0].message.content


In [ ]:
# ========== 在笔记本中渲染 Markdown ==========

# display(Markdown(...))：把摘要当 Markdown 显示（而不是纯文本 print）
display(Markdown(summary))
